# T2 Structured Information Extraction — broad/vague field output for T3/T4

This version follows the T1 → T2 → T3/T4 pipeline and saves the final output in the same flat schema as `extracted_claims.jsonl`.

Main changes in this broad version:
- It is **less strict** about field values, especially `dataset`, `metric`, `model`, and `comparison`.
- It keeps vague but explicitly stated values such as patient groups, cohorts, samples, biological outcomes, and study groups.
- It still does **not hallucinate**: values are only copied or derived from explicit text patterns.
- It saves only two files: `t2_results_summary.json` and `t2_structured_claims.jsonl`.

Final flat output schema:

```json
{"paper_id":"DOI","sentence_id":11,"title":"...","year":2025,"claim_text":"...","raw_sentence":"...","text":"[2025] Title | claim","model":null,"task":null,"dataset":null,"metric":null,"value":null,"comparison":null,"domain":null}
```


In [1]:

from pathlib import Path
import json, re, copy, math, random, shutil
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

# =====================
# PATHS
# =====================
DATA_DIR = Path('/work/Research/MedClaimExtractor/Data')
NOTEBOOK_DIR = Path('/work/Research/MedClaimExtractor')
if not DATA_DIR.exists():
    DATA_DIR = Path('/mnt/data')
    NOTEBOOK_DIR = Path('/mnt/data')

# Broad/vague-value version: prefer the original broad files first.
# If you want the cleaned strict files, move them to the top manually.
TRAIN_FILE_CANDIDATES = [
    DATA_DIR / 'Train-Dataset.jsonl',
]
GOLD_FILE_CANDIDATES = [
    DATA_DIR / 'golden.jsonl',
]

TRAIN_FILE = next((p for p in TRAIN_FILE_CANDIDATES if p.exists()), TRAIN_FILE_CANDIDATES[0])
GOLD_FILE  = next((p for p in GOLD_FILE_CANDIDATES if p.exists()), GOLD_FILE_CANDIDATES[0])

# T1 output candidates. Add your exact T1 prediction path here if needed.
T1_PRED_FILE_CANDIDATES = [
    NOTEBOOK_DIR / 'outputs' / 't1_claim_predictions.jsonl',
    Path('outputs/t1_claim_predictions.jsonl'),
    Path('/mnt/data/t1_claim_predictions.jsonl'),
]

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 't2'
MODEL_DIR  = NOTEBOOK_DIR / 'models' / 't2' / 'best_model'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

# =====================
# CONFIG
# =====================
MODEL_NAME = 'google/flan-t5-small'  # use google/flan-t5-base if GPU memory allows
SEED = 42
GOLD_VAL_RATIO = 0.50
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128
BATCH_SIZE = 4
EPOCHS = 2
LR = 5e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

# Use more training data to avoid under-extraction/all-null behavior.
FILTER_TRAINING_ROWS = True
MIN_CORE_NONNULL = 1
REMOVE_EMPTY_TEXT = True
REMOVE_DOMAIN_ONLY_ROWS = False                  # keep more diverse data
REMOVE_VALUE_WITHOUT_METRIC_OR_TASK = False      # broad output: keep explicit value-only rows
REMOVE_WEAK_SUSPICIOUS_ROWS = False              # broad output: keep vague datasets/biological outcomes
USE_RULE_BASED_TARGET_REPAIR = True
USE_RULE_BASED_PRED_REPAIR = True
CLEAR_PREVIOUS_MODEL = True

REQUIRED_FIELDS = ['model','task','dataset','metric','value','comparison','domain']
CORE_FIELDS = ['model','task','dataset','metric','value','comparison']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', DEVICE)
print('TRAIN_FILE:', TRAIN_FILE, TRAIN_FILE.exists())
print('GOLD_FILE:', GOLD_FILE, GOLD_FILE.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)


Device: cpu
TRAIN_FILE: /work/Research/MedClaimExtractor/Data/Train-Dataset.jsonl True
GOLD_FILE: /work/Research/MedClaimExtractor/Data/golden.jsonl True
OUTPUT_DIR: /work/Research/MedClaimExtractor/outputs/t2


/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:

# =====================
# UTILITIES
# =====================
def read_jsonl(path):
    path = Path(path)
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f'JSON error in {path} line {i}: {e}')
    return rows

def write_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print('Saved:', path)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print('Saved:', path)

def is_nonnull(v):
    if v is None:
        return False
    if isinstance(v, float) and math.isnan(v):
        return False
    if isinstance(v, str) and v.strip().lower() in ['', 'null', 'none', 'nan']:
        return False
    if isinstance(v, (list, dict)) and len(v) == 0:
        return False
    return True

def normalize_si(si):
    if not isinstance(si, dict):
        si = {}
    return {k: (si.get(k) if is_nonnull(si.get(k)) else None) for k in REQUIRED_FIELDS}

def paper_key(p):
    return str(p.get('doi') or p.get('paper_id') or p.get('title') or '').strip()

def sentence_text(s):
    return str(s.get('text') or s.get('sentence') or s.get('raw_sentence') or '').strip()

def flatten_claim_rows(papers, source_name):
    rows = []
    for p in papers:
        doi = str(p.get('doi') or p.get('paper_id') or '').strip()
        title = str(p.get('title') or '').strip()
        year = p.get('year')
        for s in p.get('sentences', []):
            try:
                label = int(s.get('label', 0))
            except Exception:
                label = 0
            if label != 1:
                continue
            text = sentence_text(s)
            si = normalize_si(s.get('structured_info', {}))
            rows.append({
                'source': source_name,
                'paper_id': doi or title,
                'doi': doi,
                'title': title,
                'year': year,
                'sentence_id': s.get('sentence_id'),
                'text': text,
                'target': si,
                'structured_info': si,
            })
    return rows

def count_nonnull_fields(rows, name):
    c = Counter()
    all_null = 0
    for r in rows:
        si = normalize_si(r.get('structured_info') or r.get('target') or {})
        anyv = False
        for f in REQUIRED_FIELDS:
            if is_nonnull(si.get(f)):
                c[f] += 1
                anyv = True
        if not anyv:
            all_null += 1
    print(f'\n{name} structured_info sanity check')
    print('  rows:', len(rows))
    print('  all-null rows:', all_null)
    print('  non-null counts:', dict(c))
    return c


In [3]:

# =====================
# LOAD DATA + GOLDEN PAPER SPLIT
# =====================
train_papers = read_jsonl(TRAIN_FILE)
golden_papers = read_jsonl(GOLD_FILE)

train_rows_all = flatten_claim_rows(train_papers, 'train_llm')
golden_rows_all = flatten_claim_rows(golden_papers, 'golden_manual')

print(f'LLM training claim sentences: {len(train_rows_all)} | papers: {len(train_papers)}')
print(f'Golden claim sentences:       {len(golden_rows_all)} | papers: {len(golden_papers)}')
count_nonnull_fields(train_rows_all, 'TRAIN BEFORE FILTER')
count_nonnull_fields(golden_rows_all, 'GOLDEN ALL')

idxs = list(range(len(golden_papers)))
val_idx, test_idx = train_test_split(idxs, test_size=1-GOLD_VAL_RATIO, random_state=SEED)
val_papers = [golden_papers[i] for i in val_idx]
test_papers = [golden_papers[i] for i in test_idx]

gold_val_rows = flatten_claim_rows(val_papers, 'golden_val')
gold_test_rows = flatten_claim_rows(test_papers, 'golden_test')

split_summary = {
    'train_file': str(TRAIN_FILE),
    'gold_file': str(GOLD_FILE),
    'train_claim_rows': len(train_rows_all),
    'train_papers': len(train_papers),
    'golden_claim_rows': len(golden_rows_all),
    'golden_papers': len(golden_papers),
    'gold_val_rows': len(gold_val_rows),
    'gold_val_papers': len(val_papers),
    'gold_test_rows': len(gold_test_rows),
    'gold_test_papers': len(test_papers),
    'seed': SEED,
    'gold_val_ratio': GOLD_VAL_RATIO,
}
save_json(split_summary, OUTPUT_DIR / 't2_data_split_summary.json')
print(split_summary)


LLM training claim sentences: 4172 | papers: 1269
Golden claim sentences:       1484 | papers: 316

TRAIN BEFORE FILTER structured_info sanity check
  rows: 4172
  all-null rows: 0
  non-null counts: {'task': 2578, 'dataset': 500, 'metric': 1427, 'value': 1017, 'comparison': 1139, 'domain': 3111, 'model': 759}

GOLDEN ALL structured_info sanity check
  rows: 1484
  all-null rows: 0
  non-null counts: {'task': 977, 'domain': 954, 'comparison': 677, 'model': 471, 'dataset': 285, 'metric': 925, 'value': 345}
Saved: /work/Research/MedClaimExtractor/outputs/t2/t2_data_split_summary.json
{'train_file': '/work/Research/MedClaimExtractor/Data/Train-Dataset.jsonl', 'gold_file': '/work/Research/MedClaimExtractor/Data/golden.jsonl', 'train_claim_rows': 4172, 'train_papers': 1269, 'golden_claim_rows': 1484, 'golden_papers': 316, 'gold_val_rows': 736, 'gold_val_papers': 158, 'gold_test_rows': 748, 'gold_test_papers': 158, 'seed': 42, 'gold_val_ratio': 0.5}


In [4]:

# =====================
# BROAD FILTERING + RULE-BASED TARGET REPAIR
# =====================
# This prompt is intentionally less strict than the publication-clean version.
# It allows vague but explicit values, similar to extracted_claims.jsonl.
PROMPT_TMPL = "\n".join([
    "Extract structured JSON from the sentence.",
    "Use null only if the field is not stated in the sentence.",
    "Fields: model, task, dataset, metric, value, comparison, domain.",
    "Allow explicitly stated patient groups, cohorts, samples, biological outcomes, biomarkers, drugs, models, methods, metrics, values, and comparisons.",
    "Examples: dataset=glioma patients, metric=cell proliferation, AUC=0.93, accuracy=0.94, sensitivity=0.916, p-value=0.04.",
    "Sentence: {claim}",
    "JSON:"
])

# Named + vague patterns. These are used only when the text explicitly contains them.
DATASET_PATTERNS = [
    r'\bTCGA(?:-[A-Z]+)?\b', r'\bCGGA\b', r'\bBraTS(?:\s*\d{4})?\b', r'\bUKB\b',
    r'UK Biobank', r'\bFinnGen\b', r'\bGEO\b', r'\bGSE\d+\b', r'\bFigshare\b',
    r'\bcBioPortal\b', r'\bTCIA\b', r'\bUCSF[^,.;)]*(?:Study|cohort)?',
    r'\b\d+\s+(?:patients|cases|samples|studies|subjects)\b',
    r'\b(?:glioma|GBM|LGG|glioblastoma|brain tumor|meningioma|metastases?)\s+(?:patients|cases|samples|cells|cohort|tissues|specimens)\b',
    r'\b(?:training|validation|test|study|discovery|internal|external)\s+cohort\b',
    r'\b(?:included|eligible)\s+studies\b',
    r'\bpatient-derived\s+(?:cells|samples|models)\b',
]

MODEL_PATTERNS = [
    r'\bResNet\d*\b', r'\bDenseNet\d*\b', r'\bVGG\d*\b', r'\bU-Net\b', r'\bnnU-Net\b',
    r'\bSVM\b', r'\brandom forest\b', r'\belastic net\b', r'\bneural network\b', r'\bCNN\b',
    r'\bDCNN\b', r'\bGAN\b', r'\bBiGAN\b', r'\bGoogLeNet\b', r'\bViT\w*\b', r'\bDeiT\b', r'\bBeiT\b',
    r'\bnomogram\b', r'\bCox model\b', r'\brisk score\b', r'\bradiomics signature\b', r'\bgene signature\b',
    r'\bsignature\b', r'\bclassifier\b', r'\bhybrid ensemble\b', r'\bcustom CNN\b',
    # Broad entities often present in extracted_claims-style outputs
    r'\b[A-Z0-9]{2,}(?:-[A-Z0-9]+)?\b',
]

FORMAL_METRIC_PATTERNS = {
    'AUC': [r'\bAUC\b', r'area under the curve'],
    'accuracy': [r'\baccuracy\b'],
    'sensitivity': [r'\bsensitivity\b(?!\s+analysis)'],
    'specificity': [r'\bspecificity\b'],
    'Dice': [r'\bDice\b', r'\bDSC\b'],
    'F1-score': [r'\bF1\b', r'F1-score'],
    'precision': [r'\bprecision\b'],
    'recall': [r'\brecall\b'],
    'p-value': [r'\bp\s*[<=>]\s*[0-9.]', r'\bp-value\b'],
    'hazard ratio': [r'\bHR\b', r'hazard ratio'],
    'odds ratio': [r'\bOR\b', r'odds ratio'],
    'confidence interval': [r'\bCI\b', r'confidence interval'],
    'R2': [r'\bR2\b', r'R\^2'],
    'C-index': [r'\bC-index\b', r'concordance index'],
}

BROAD_METRIC_TERMS = [
    'cell proliferation', 'colony formation', 'wound healing', 'migration', 'angiogenesis',
    'protein expression', 'gene expression', 'expression', 'pathway activity', 'pathway enrichment',
    'immune infiltration', 'tumor grade', 'survival', 'overall survival', 'progression-free survival',
    'functional connectivity', 'stemness', 'glycolysis', 'therapeutic response', 'treatment response',
    'drug sensitivity', 'tumor volume', 'recurrence', 'diagnostic performance', 'classification performance',
    'consistency of imaging features', 'firmness', 'risk', 'association', 'correlation', 'incidence',
]

TASK_RULES = {
    'segmentation': [r'\bsegmentation\b', r'\bsegment(?:ed|ing)?\b'],
    'classification': [r'\bclassification\b', r'\bclassif(?:y|ied|ication)\b'],
    'detection': [r'\bdetection\b', r'\bdetect(?:ed|ion)?\b'],
    'diagnosis': [r'\bdiagnos(?:is|tic|e)\b'],
    'survival prediction': [r'\bsurvival\b', r'overall survival', r'\bOS\b'],
    'recurrence detection': [r'\brecurrence\b', r'\brecurrent\b'],
    'grading': [r'\bgrading\b', r'\bgrade\b'],
    'biomarker identification': [r'\bbiomarker\b'],
    'mutation analysis': [r'\bmutation\b', r'\bmutations\b'],
    'pathway enrichment analysis': [r'pathway enrichment', r'GO enrichment', r'KEGG'],
    'treatment response prediction': [r'treatment response', r'therapeutic response'],
    'correlation analysis': [r'\bcorrelat(?:ion|ed)\b'],
    'risk association analysis': [r'\brisk\b', r'\bassociat(?:ion|ed)\b'],
}

DOMAIN_RULES = {
    'brain tumor': [r'brain tumors?', r'brain neoplasms?'],
    'brain tumor segmentation': [r'brain tumor.*segmentation', r'segmentation.*brain tumor'],
    'brain tumor classification': [r'brain tumor.*classification', r'classification.*brain tumor'],
    'brain tumor detection': [r'brain tumor.*detection', r'detection.*brain tumor'],
    'glioma': [r'\bglioma\b', r'\bgliomas\b'],
    'glioblastoma': [r'\bglioblastoma\b', r'\bGBM\b'],
    'LGG': [r'\bLGG\b', r'low-grade glioma'],
    'meningioma': [r'\bmeningioma\b'],
    'metastasis': [r'\bmetastases\b', r'\bmetastasis\b'],
}

COMPARISON_PATTERNS = [
    r'compared\s+(?:with|to)\s+([^.;,]+)',
    r'versus\s+([^.;,]+)',
    r'\bvs\.?\s+([^.;,]+)',
    r'(?:higher|lower|better|worse|superior|inferior)\s+than\s+([^.;,]+)',
    r'outperform(?:ed|s)?\s+([^.;,]+)',
    r'between\s+([^.;]+?\s+and\s+[^.;]+)',
    r'while\s+([^.;]+)',
]

def only_domain_filled(si):
    si = normalize_si(si)
    return is_nonnull(si.get('domain')) and all(not is_nonnull(si.get(f)) for f in CORE_FIELDS)

def value_without_metric_or_task(si):
    si = normalize_si(si)
    return is_nonnull(si.get('value')) and not is_nonnull(si.get('metric')) and not is_nonnull(si.get('task'))

def is_bad_training_row(r):
    si = normalize_si(r.get('structured_info', {}))
    text = str(r.get('text') or '').strip()
    if REMOVE_EMPTY_TEXT and not text:
        return True, 'empty_text'
    if FILTER_TRAINING_ROWS and sum(is_nonnull(si.get(f)) for f in CORE_FIELDS) < MIN_CORE_NONNULL:
        return True, 'too_few_core_fields'
    if REMOVE_DOMAIN_ONLY_ROWS and only_domain_filled(si):
        return True, 'domain_only'
    if REMOVE_VALUE_WITHOUT_METRIC_OR_TASK and value_without_metric_or_task(si):
        return True, 'value_without_metric_or_task'
    # Broad mode: do not remove vague datasets, broad metrics, or gene/drug-like model values.
    return False, 'keep'

def pct_to_decimal(x):
    return float(x) / 100.0

def format_decimal(x):
    return f'{x:.4g}'

def unique_join(items):
    vals = []
    for x in items:
        if x is None:
            continue
        s = str(x).strip(' ,.;:()[]')
        if s and s.lower() not in [v.lower() for v in vals]:
            vals.append(s)
    return '; '.join(vals) if vals else None

def regex_find_all(patterns, text):
    found = []
    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.I):
            if m.groups():
                found.append(m.group(1))
            else:
                found.append(m.group(0))
    return found

def broad_rule_repair_from_text(text, si):
    """Broad extractor/repairer: keeps vague but explicit values similar to extracted_claims.jsonl."""
    si = normalize_si(copy.deepcopy(si) if isinstance(si, dict) else {})
    text = str(text or '')
    text_l = text.lower()

    # model/method/entity
    if not is_nonnull(si.get('model')):
        models = regex_find_all(MODEL_PATTERNS, text)
        # Avoid using very generic all-caps words as model.
        generic = {'MRI','PET','CT','DNA','RNA','GBM','LGG','CNS','WHO','AUC','CI','HR','OR','OS','PFS'}
        models = [m for m in models if str(m).strip().upper() not in generic]
        if models:
            si['model'] = unique_join(models[:5])

    # dataset/source/cohort — named or vague explicitly stated values
    if not is_nonnull(si.get('dataset')):
        datasets = regex_find_all(DATASET_PATTERNS, text)
        if datasets:
            si['dataset'] = unique_join(datasets[:5])

    # task
    if not is_nonnull(si.get('task')):
        tasks = []
        for task, pats in TASK_RULES.items():
            if any(re.search(p, text, flags=re.I) for p in pats):
                tasks.append(task)
        if tasks:
            si['task'] = unique_join(tasks[:2])

    # domain
    if not is_nonnull(si.get('domain')):
        domains = []
        for dom, pats in DOMAIN_RULES.items():
            if any(re.search(p, text, flags=re.I) for p in pats):
                domains.append(dom)
        if domains:
            # Prefer more specific domains if available.
            domains = sorted(domains, key=len, reverse=True)
            si['domain'] = domains[0]

    # formal metric + broad metric/outcome terms
    if not is_nonnull(si.get('metric')):
        metrics = []
        for metric, pats in FORMAL_METRIC_PATTERNS.items():
            if any(re.search(p, text, flags=re.I) for p in pats):
                metrics.append(metric)
        for term in BROAD_METRIC_TERMS:
            if re.search(r'\b' + re.escape(term) + r'\b', text_l):
                metrics.append(term)
        if metrics:
            si['metric'] = unique_join(metrics[:6])

    # values: percentages, AUC decimals, p-values, HR/OR/CI, plain paired values
    if not is_nonnull(si.get('value')):
        value_parts = []
        # metric percent pairs
        for m in re.finditer(r'\b(accuracy|sensitivity|specificity|precision|recall|F1(?:-score)?|Dice|DSC)\b[^0-9]{0,20}([0-9]+(?:\.[0-9]+)?)\s*%', text, flags=re.I):
            value_parts.append(f'{m.group(1)} = {format_decimal(pct_to_decimal(m.group(2)))}')
        # AUC decimals
        for m in re.finditer(r'\bAUC\s*(?:=|of|:)?\s*([0-9]*\.?[0-9]+)', text, flags=re.I):
            value_parts.append(f'AUC = {m.group(1)}')
        # p-values
        for m in re.finditer(r'\b[Pp]\s*([<=>])\s*([0-9]*\.?[0-9]+(?:E-?\d+)?)', text):
            value_parts.append(f'P {m.group(1)} {m.group(2)}')
        # HR/OR
        for m in re.finditer(r'\b(HR|OR)\s*(?:=|:)\s*([0-9]*\.?[0-9]+)', text, flags=re.I):
            value_parts.append(f'{m.group(1).upper()} = {m.group(2)}')
        # CI ranges
        for m in re.finditer(r'(?:95%\s*)?CI\s*(?:=|:)?\s*([0-9.]+\s*[-–]\s*[0-9.]+)', text, flags=re.I):
            value_parts.append(f'95% CI = {m.group(1).replace("–", "-")}')
        # If no pair is found, keep a compact list of explicit percentages (useful for comparisons like SVM 96%, VGG 94%).
        if not value_parts:
            pct_vals = re.findall(r'~?([0-9]+(?:\.[0-9]+)?)\s*%', text)
            if pct_vals:
                value_parts = [str(format_decimal(pct_to_decimal(x))) for x in pct_vals[:10]]
        if value_parts:
            si['value'] = '; '.join(dict.fromkeys(value_parts))

    # comparison
    if not is_nonnull(si.get('comparison')):
        comps = regex_find_all(COMPARISON_PATTERNS, text)
        if comps:
            si['comparison'] = unique_join(comps[:3])

    return normalize_si(si)

# Apply minimal filters and broad target repair
train_rows = []
reason_counts = Counter()
for r in train_rows_all:
    bad, reason = is_bad_training_row(r)
    reason_counts[reason] += 1
    if bad:
        continue
    rr = copy.deepcopy(r)
    if USE_RULE_BASED_TARGET_REPAIR:
        rr['structured_info'] = broad_rule_repair_from_text(rr.get('text', ''), rr.get('structured_info', {}))
        rr['target'] = rr['structured_info']
    train_rows.append(rr)

print('Training filter reason counts:', dict(reason_counts))
print('Train rows after broad filtering/repair:', len(train_rows))
count_nonnull_fields(train_rows, 'TRAIN AFTER BROAD FILTER/REPAIR')

# No audit CSVs are saved in this clean-output version.
def to_target_json(d):
    return json.dumps(normalize_si(d), ensure_ascii=False)

def make_pairs(rows, name):
    rows = [r for r in rows if str(r.get('text') or '').strip()]
    inputs = [PROMPT_TMPL.format(claim=r['text']) for r in rows]
    targets = [to_target_json(r.get('target') or r.get('structured_info') or {}) for r in rows]
    print(f'{name}: rows = {len(rows)}')
    return inputs, targets, rows

train_inputs, train_targets, train_rows = make_pairs(train_rows, 'TRAIN')
val_inputs, val_targets, gold_val_rows = make_pairs(gold_val_rows, 'GOLD VAL')
test_inputs, test_targets, gold_test_rows = make_pairs(gold_test_rows, 'GOLD TEST')

print('\nExample input:')
print(train_inputs[0] if train_inputs else 'NO TRAIN INPUT')
print('\nExample target:')
print(train_targets[0] if train_targets else 'NO TRAIN TARGET')


Training filter reason counts: {'keep': 3536, 'too_few_core_fields': 636}
Train rows after broad filtering/repair: 3536

TRAIN AFTER BROAD FILTER/REPAIR structured_info sanity check
  rows: 3536
  all-null rows: 0
  non-null counts: {'model': 3536, 'task': 2839, 'dataset': 725, 'metric': 2100, 'value': 1206, 'comparison': 1291, 'domain': 2500}
TRAIN: rows = 3536
GOLD VAL: rows = 736
GOLD TEST: rows = 748

Example input:
Extract structured JSON from the sentence.
Use null only if the field is not stated in the sentence.
Fields: model, task, dataset, metric, value, comparison, domain.
Allow explicitly stated patient groups, cohorts, samples, biological outcomes, biomarkers, drugs, models, methods, metrics, values, and comparisons.
Examples: dataset=glioma patients, metric=cell proliferation, AUC=0.93, accuracy=0.94, sensitivity=0.916, p-value=0.04.
Sentence: Results: There was a positive correlation among firefighters between the median number of sample variants attributable to single-ba

In [5]:

# =====================
# DATASET CREATION
# =====================
def to_target_json(si):
    return json.dumps(normalize_si(si), ensure_ascii=False)

def make_pairs(rows, name):
    clean = [r for r in rows if (not REMOVE_EMPTY_TEXT or str(r.get('text') or '').strip())]
    inputs = [PROMPT_TMPL.format(claim=r['text']) for r in clean]
    targets = [to_target_json(r.get('structured_info') or r.get('target') or {}) for r in clean]
    print(f'{name}: rows={len(clean)}')
    return inputs, targets, clean

train_inputs, train_targets, train_rows = make_pairs(train_rows, 'TRAIN')
val_inputs, val_targets, gold_val_rows = make_pairs(gold_val_rows, 'GOLD VAL')
test_inputs, test_targets, gold_test_rows = make_pairs(gold_test_rows, 'GOLD TEST')

print('\nRaw first training text:')
print(repr(train_rows[0]['text']))
print('\nExample input:')
print(train_inputs[0])
print('\nExample target:')
print(train_targets[0])

class S2SDataset(Dataset):
    def __init__(self, inputs, targets, tokenizer):
        self.inputs = inputs
        self.targets = targets
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, idx):
        x = self.tokenizer(self.inputs[idx], max_length=MAX_INPUT_LEN, truncation=True, padding='max_length', return_tensors='pt')
        y = self.tokenizer(self.targets[idx], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length', return_tensors='pt')
        labels = y['input_ids'].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            'input_ids': x['input_ids'].squeeze(0),
            'attention_mask': x['attention_mask'].squeeze(0),
            'labels': labels,
        }


TRAIN: rows=3536
GOLD VAL: rows=736
GOLD TEST: rows=748

Raw first training text:
'Results: There was a positive correlation among firefighters between the median number of sample variants attributable to single-base substitution signature 42, a single-base substitution mutational signature associated with haloalkane exposure (from the Catalogue of Somatic Mutational Signatures in Cancer) and firefighting years (p = .04; R2 = 0.29).'

Example input:
Extract structured JSON from the sentence.
Use null only if the field is not stated in the sentence.
Fields: model, task, dataset, metric, value, comparison, domain.
Allow explicitly stated patient groups, cohorts, samples, biological outcomes, biomarkers, drugs, models, methods, metrics, values, and comparisons.
Examples: dataset=glioma patients, metric=cell proliferation, AUC=0.93, accuracy=0.94, sensitivity=0.916, p-value=0.04.
Sentence: Results: There was a positive correlation among firefighters between the median number of sample vari

In [6]:

# =====================
# PARSING, BROAD POST-PROCESSING, METRICS
# =====================
def parse_jsonish(s):
    s = str(s).strip()
    if not s:
        return normalize_si({})
    s = re.sub(r'^```(?:json)?', '', s.strip(), flags=re.I).strip()
    s = re.sub(r'```$', '', s.strip()).strip()
    candidates = [s]
    if not s.startswith('{'):
        candidates.append('{' + s + '}')
    for cand in candidates:
        try:
            return normalize_si(json.loads(cand))
        except Exception:
            pass
    out = {}
    for key in REQUIRED_FIELDS:
        m = re.search(rf'"?{key}"?\s*:\s*(null|None|"[^"]*"|[^,}}]+)', s, flags=re.I)
        if m:
            val = m.group(1).strip()
            if val.lower() in ['null', 'none']:
                out[key] = None
            elif val.startswith('"') and val.endswith('"'):
                out[key] = val[1:-1]
            else:
                out[key] = val.strip()
    return normalize_si(out)

def rule_postprocess_prediction(text, pred):
    # Broad mode: apply the same repair/extraction rules to model predictions.
    return broad_rule_repair_from_text(text, pred)

def generate_predictions(model, inputs, batch_size=8, rows_for_repair=None):
    model.eval()
    raw = []
    preds = []
    for i in tqdm(range(0, len(inputs), batch_size), desc='Generate'):
        batch_inputs = inputs[i:i+batch_size]
        enc = tokenizer(batch_inputs, max_length=MAX_INPUT_LEN, truncation=True, padding=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_TARGET_LEN, num_beams=4, early_stopping=True)
        outs = tokenizer.batch_decode(gen, skip_special_tokens=True)
        raw.extend(outs)
        for j, out in enumerate(outs):
            pred = parse_jsonish(out)
            if USE_RULE_BASED_PRED_REPAIR and rows_for_repair is not None:
                row = rows_for_repair[i+j]
                pred = rule_postprocess_prediction(row.get('text') or row.get('claim_text') or '', pred)
            preds.append(normalize_si(pred))
    return raw, preds

def norm_compare_value(x):
    if not is_nonnull(x):
        return None
    return str(x).strip().lower()

def field_metrics(golds, preds):
    rows = []
    total = len(golds)
    for f in REQUIRED_FIELDS:
        gvals = [g.get(f) for g in golds]
        pvals = [p.get(f) for p in preds]
        ge = [is_nonnull(x) for x in gvals]
        pe = [is_nonnull(x) for x in pvals]
        exact = sum(norm_compare_value(gvals[i]) == norm_compare_value(pvals[i]) for i in range(total)) / total if total else 0
        tp = sum(ge[i] and pe[i] for i in range(total))
        fp = sum((not ge[i]) and pe[i] for i in range(total))
        fn = sum(ge[i] and (not pe[i]) for i in range(total))
        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
        null_idx = [i for i, x in enumerate(ge) if not x]
        non_idx = [i for i, x in enumerate(ge) if x]
        null_acc = sum((not pe[i]) for i in null_idx) / len(null_idx) if null_idx else 0
        non_exact = sum(norm_compare_value(gvals[i]) == norm_compare_value(pvals[i]) for i in non_idx) / len(non_idx) if non_idx else 0
        rows.append({
            'field': f,
            'accuracy': exact,
            'precision_field_exists': precision,
            'recall_field_exists': recall,
            'f1_field_exists': f1,
            'null_accuracy': null_acc,
            'non_null_exact_accuracy': non_exact,
            'gold_non_null_count': sum(ge),
            'pred_non_null_count': sum(pe),
            'total_examples': total,
        })
    return pd.DataFrame(rows)

def evaluate_generation(model, rows, inputs, split_name):
    raw, preds = generate_predictions(model, inputs, batch_size=max(1, BATCH_SIZE*2), rows_for_repair=rows)
    golds = [normalize_si(r.get('structured_info') or r.get('target') or {}) for r in rows]
    fm = field_metrics(golds, preds)
    overall = {
        'split': split_name,
        'examples': len(rows),
        'json_exact_match_accuracy': float(sum(g == p for g, p in zip(golds, preds)) / len(golds)) if golds else 0,
        'macro_field_accuracy': float(fm['accuracy'].mean()),
        'macro_field_f1_exists': float(fm['f1_field_exists'].mean()),
        'macro_non_null_exact_accuracy': float(fm['non_null_exact_accuracy'].mean()),
    }
    return raw, preds, fm, overall

def eval_loss(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            losses.append(float(out.loss.detach().cpu()))
    return float(np.mean(losses)) if losses else None


In [7]:

# =====================
# TRAINING
# =====================
if CLEAR_PREVIOUS_MODEL and MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
    print('Deleted old model:', MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

train_ds = S2SDataset(train_inputs, train_targets, tokenizer)
val_ds = S2SDataset(val_inputs, val_targets, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.05 * total_steps)),
    num_training_steps=total_steps,
)

best_val_f1 = -1
best_record = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        out = model(**batch)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        losses.append(float(loss.detach().cpu()))
        pbar.set_postfix(loss=np.mean(losses))

    train_loss = float(np.mean(losses))
    val_loss = eval_loss(model, val_loader)
    _, _, val_fm, val_overall = evaluate_generation(model, gold_val_rows, val_inputs, 'gold_validation')
    val_f1 = val_overall['macro_field_f1_exists']

    rec = {
        'epoch': epoch,
        'train_loss': train_loss,
        'gold_val_loss': val_loss,
        'gold_val_macro_presence_f1': val_f1,
        'gold_val_json_exact_match_accuracy': val_overall['json_exact_match_accuracy'],
        'gold_val_macro_non_null_exact_accuracy': val_overall['macro_non_null_exact_accuracy'],
    }
    history.append(rec)

    print('\nEpoch', epoch, 'train_loss', train_loss, 'val_loss', val_loss, 'val_macro_f1', val_f1)
    print(val_fm[['field', 'f1_field_exists', 'gold_non_null_count', 'pred_non_null_count']])

    # Save best model by validation Macro Presence F1, not only loss.
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_record = rec
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print('✓ Saved best model:', MODEL_DIR)


Deleted old model: /work/Research/MedClaimExtractor/models/t2/best_model


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Epoch 1/2:   0%|          | 0/884 [00:00<?, ?it/s]

Generate:   0%|          | 0/92 [00:00<?, ?it/s]


Epoch 1 train_loss 1.0265549825529707 val_loss 0.873467597057638 val_macro_f1 0.5006184804062865
        field  f1_field_exists  gold_non_null_count  pred_non_null_count
0       model         0.450526                  214                  736
1        task         0.531579                  487                  273
2     dataset         0.241206                  131                   68
3      metric         0.559783                  447                  289
4       value         0.731861                  171                  146
5  comparison         0.372294                  355                  107
6      domain         0.617080                  479                  247
✓ Saved best model: /work/Research/MedClaimExtractor/models/t2/best_model


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 2/2:   0%|          | 0/884 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Generate:   0%|          | 0/92 [00:00<?, ?it/s]


Epoch 2 train_loss 0.6744581235075429 val_loss 0.8487380979501683 val_macro_f1 0.5056053860178006
        field  f1_field_exists  gold_non_null_count  pred_non_null_count
0       model         0.450526                  214                  736
1        task         0.540962                  487                  282
2     dataset         0.241206                  131                   68
3      metric         0.569501                  447                  294
4       value         0.738462                  171                  154
5  comparison         0.375810                  355                  108
6      domain         0.622771                  479                  250
✓ Saved best model: /work/Research/MedClaimExtractor/models/t2/best_model


In [8]:

# =====================
# FINAL EVALUATION + THESIS-STYLE RESULTS TABLES
# Saves only one compact results JSON file.
# =====================
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

field_metrics_by_split = {}
overall_metrics_by_split = {}

for split_name, rows, inputs in [
    ('golden_validation', gold_val_rows, val_inputs),
    ('golden_test', gold_test_rows, test_inputs),
]:
    raw, preds, fm, overall = evaluate_generation(model, rows, inputs, split_name)
    field_metrics_by_split[split_name] = fm.copy()
    overall_metrics_by_split[split_name] = overall

val_overall = overall_metrics_by_split['golden_validation']
test_overall = overall_metrics_by_split['golden_test']
test_fm = field_metrics_by_split['golden_test'].copy()

# -------------------------------------------------------------------
# Table 1: Thesis-style overall summary: Validation vs Test
# -------------------------------------------------------------------
overall_table = pd.DataFrame([
    {
        'Metric': 'Overall exact match',
        'Validation': val_overall.get('json_exact_match_accuracy'),
        'Test': test_overall.get('json_exact_match_accuracy'),
    },
    {
        'Metric': 'Macro presence F1',
        'Validation': val_overall.get('macro_field_f1_exists'),
        'Test': test_overall.get('macro_field_f1_exists'),
    },
    {
        'Metric': 'Macro field accuracy',
        'Validation': val_overall.get('macro_field_accuracy'),
        'Test': test_overall.get('macro_field_accuracy'),
    },
    {
        'Metric': 'Macro non-null exact accuracy',
        'Validation': val_overall.get('macro_non_null_exact_accuracy'),
        'Test': test_overall.get('macro_non_null_exact_accuracy'),
    },
])

# -------------------------------------------------------------------
# Table 2: Thesis-style test field table: Exact/Non-null/Null accuracy
# -------------------------------------------------------------------
test_accuracy_table = test_fm[[
    'field', 'accuracy', 'non_null_exact_accuracy', 'null_accuracy'
]].copy()
test_accuracy_table['field'] = test_accuracy_table['field'].str.title()
test_accuracy_table = test_accuracy_table.rename(columns={
    'field': 'Field',
    'accuracy': 'Exact Match',
    'non_null_exact_accuracy': 'Non-Null Accuracy',
    'null_accuracy': 'Null Accuracy',
})

# -------------------------------------------------------------------
# Table 3: Test-only precision, recall, accuracy and F1
# -------------------------------------------------------------------
test_prf_table = test_fm[[
    'field', 'accuracy', 'precision_field_exists', 'recall_field_exists', 'f1_field_exists'
]].copy()
test_prf_table['field'] = test_prf_table['field'].str.title()
test_prf_table = test_prf_table.rename(columns={
    'field': 'Field',
    'accuracy': 'Accuracy',
    'precision_field_exists': 'Precision',
    'recall_field_exists': 'Recall',
    'f1_field_exists': 'F1',
})

# Round only for display. JSON keeps full precision below.
display_overall = overall_table.copy()
for c in ['Validation', 'Test']:
    display_overall[c] = display_overall[c].map(lambda x: round(float(x), 4) if x is not None else None)

display_accuracy = test_accuracy_table.copy()
for c in ['Exact Match', 'Non-Null Accuracy', 'Null Accuracy']:
    display_accuracy[c] = display_accuracy[c].map(lambda x: round(float(x), 4) if x is not None else None)

display_prf = test_prf_table.copy()
for c in ['Accuracy', 'Precision', 'Recall', 'F1']:
    display_prf[c] = display_prf[c].map(lambda x: round(float(x), 4) if x is not None else None)

print('\n' + '=' * 90)
print('OVERALL T2 RESULTS: GOLDEN VALIDATION VS GOLDEN TEST')
print('=' * 90)
display(display_overall)

print('\n' + '=' * 90)
print('TEST FIELD RESULTS: EXACT MATCH, NON-NULL ACCURACY, NULL ACCURACY')
print('=' * 90)
display(display_accuracy)

print('\n' + '=' * 90)
print('TEST FIELD RESULTS: ACCURACY, PRECISION, RECALL, F1')
print('=' * 90)
display(display_prf)

# Save only compact results JSON, no separate CSV/prediction files.
results_summary = {
    'model_dir': str(MODEL_DIR),
    'train_file': str(TRAIN_FILE),
    'gold_file': str(GOLD_FILE),
    'best_val_macro_presence_f1': globals().get('best_val_f1', None),
    'best_record': globals().get('best_record', None),
    'overall_table': overall_table.to_dict(orient='records'),
    'test_accuracy_table': test_accuracy_table.to_dict(orient='records'),
    'test_precision_recall_f1_table': test_prf_table.to_dict(orient='records'),
    'golden_validation_overall': val_overall,
    'golden_test_overall': test_overall,
    'golden_test_field_metrics_full': test_fm.to_dict(orient='records'),
}
save_json(results_summary, OUTPUT_DIR / 't2_results_summary.json')
print('\nSaved only compact results summary:')
print(OUTPUT_DIR / 't2_results_summary.json')


Generate:   0%|          | 0/92 [00:00<?, ?it/s]

Generate:   0%|          | 0/94 [00:00<?, ?it/s]


OVERALL T2 RESULTS: GOLDEN VALIDATION VS GOLDEN TEST


,Metric,Validation,Test
0,Overall exact match,0.0000,0.0013
1,Macro presence F1,0.5056,0.5202
2,Macro field accuracy,0.4284,0.4282
3,Macro non-null exact accuracy,0.0485,0.0506



TEST FIELD RESULTS: EXACT MATCH, NON-NULL ACCURACY, NULL ACCURACY


,Field,Exact Match,Non-Null Accuracy,Null Accuracy
0,Model,0.0040,0.0117,0.0000
1,Task,0.2500,0.0245,0.6783
2,Dataset,0.7313,0.0390,0.9108
3,Metric,0.3128,0.0921,0.7037
4,Value,0.7366,0.0460,0.9460
5,Comparison,0.5481,0.0124,0.9531
6,Domain,0.4144,0.1284,0.9121



TEST FIELD RESULTS: ACCURACY, PRECISION, RECALL, F1


,Field,Accuracy,Precision,Recall,F1
0,Model,0.0040,0.3436,1.0000,0.5114
1,Task,0.2500,0.7046,0.4041,0.5136
2,Dataset,0.7313,0.3765,0.2078,0.2678
3,Metric,0.3128,0.7173,0.4247,0.5335
4,Value,0.7366,0.7947,0.6897,0.7385
5,Comparison,0.5481,0.8291,0.3012,0.4419
6,Domain,0.4144,0.9062,0.4884,0.6347


Saved: /work/Research/MedClaimExtractor/outputs/t2/t2_results_summary.json

Saved only compact results summary:
/work/Research/MedClaimExtractor/outputs/t2/t2_results_summary.json


In [9]:

# =====================
# T1 CLAIMS -> T2 STRUCTURED CLAIMS FOR T3/T4
# Saves only the final flat claim-structure JSONL file.
# Output schema exactly matches extracted_claims.jsonl:
# paper_id, sentence_id, title, year, claim_text, raw_sentence, text,
# model, task, dataset, metric, value, comparison, domain
# =====================
FINAL_FIELDS = [
    'paper_id', 'sentence_id', 'title', 'year', 'claim_text', 'raw_sentence', 'text',
    'model', 'task', 'dataset', 'metric', 'value', 'comparison', 'domain'
]

def find_t1_file():
    for p in T1_PRED_FILE_CANDIDATES:
        if Path(p).exists():
            return Path(p)
    return None

def is_predicted_claim(x):
    for key in ['predicted_label', 'pred_label', 'prediction', 't1_label', 'claim_pred', 'is_claim_pred']:
        if key in x:
            try:
                return int(x[key]) == 1
            except Exception:
                return str(x[key]).strip().lower() in ['1', 'true', 'claim', 'yes']
    for key in ['claim_score', 'score', 'prob', 'probability', 'claim_probability']:
        if key in x:
            try:
                return float(x[key]) >= 0.5
            except Exception:
                pass
    # If the T1 file already contains claim-only rows, keep them.
    return int(x.get('label', 1)) == 1

def clean_year(y):
    if y is None or str(y).strip() == '':
        return None
    try:
        return int(float(y))
    except Exception:
        return y

def make_context_text(year, title, claim_text):
    year_s = '' if year is None else str(year)
    title_s = '' if title is None else str(title).strip()
    claim_s = '' if claim_text is None else str(claim_text).strip()
    if year_s and title_s:
        return f'[{year_s}] {title_s} | {claim_s}'
    if title_s:
        return f'{title_s} | {claim_s}'
    return claim_s

def load_t1_claim_rows(t1_path):
    objs = read_jsonl(t1_path)
    rows = []

    # Case 1: paper-level JSONL with sentences list
    if objs and isinstance(objs[0], dict) and 'sentences' in objs[0]:
        for p in objs:
            doi = str(p.get('doi') or p.get('paper_id') or '').strip()
            title = str(p.get('title') or '').strip()
            year = clean_year(p.get('year'))
            for s in p.get('sentences', []):
                claim_text = sentence_text(s)
                if claim_text and is_predicted_claim(s):
                    rows.append({
                        'paper_id': doi or title,
                        'sentence_id': s.get('sentence_id'),
                        'title': title,
                        'year': year,
                        'claim_text': claim_text,
                        'raw_sentence': claim_text,
                        'text': make_context_text(year, title, claim_text),
                    })

    # Case 2: flat JSONL rows
    else:
        for r in objs:
            # Prefer clean sentence fields over the combined context field.
            claim_text = str(
                r.get('claim_text')
                or r.get('raw_sentence')
                or r.get('sentence')
                or r.get('claim')
                or r.get('text')
                or ''
            ).strip()
            if claim_text and is_predicted_claim(r):
                paper_id = str(r.get('paper_id') or r.get('doi') or '').strip()
                title = r.get('title')
                year = clean_year(r.get('year'))
                rows.append({
                    'paper_id': paper_id,
                    'sentence_id': r.get('sentence_id'),
                    'title': title,
                    'year': year,
                    'claim_text': claim_text,
                    'raw_sentence': claim_text,
                    'text': make_context_text(year, title, claim_text),
                })
    return rows

T1_PRED_FILE = find_t1_file()
print('Detected T1 prediction file:', T1_PRED_FILE)

if T1_PRED_FILE is None:
    print('No T1 prediction file found. Set T1_PRED_FILE_CANDIDATES or manually set T1_PRED_FILE and rerun this cell.')
else:
    t1_claim_rows = load_t1_claim_rows(T1_PRED_FILE)
    print('T1 predicted claim rows:', len(t1_claim_rows))

    t1_inputs = [PROMPT_TMPL.format(claim=r['claim_text']) for r in t1_claim_rows]
    raw_t1, pred_t1 = generate_predictions(
        model,
        t1_inputs,
        batch_size=max(1, BATCH_SIZE * 2),
        rows_for_repair=t1_claim_rows,
    )

    flat = []
    for r, si in zip(t1_claim_rows, pred_t1):
        claim_text = str(r.get('claim_text') or r.get('raw_sentence') or '').strip()
        year = clean_year(r.get('year'))
        title = r.get('title')
        out_row = {
            'paper_id': r.get('paper_id'),
            'sentence_id': r.get('sentence_id'),
            'title': title,
            'year': year,
            'claim_text': claim_text,
            'raw_sentence': claim_text,
            'text': make_context_text(year, title, claim_text),
            'model': si.get('model'),
            'task': si.get('task'),
            'dataset': si.get('dataset'),
            'metric': si.get('metric'),
            'value': si.get('value'),
            'comparison': si.get('comparison'),
            'domain': si.get('domain'),
        }
        # Enforce exact field order and no extra columns.
        flat.append({k: out_row.get(k) for k in FINAL_FIELDS})

    final_path = OUTPUT_DIR / 't2_structured_claims.jsonl'
    write_jsonl(flat, final_path)

    print('\nSaved final T2 claim-structure output:')
    print(final_path)
    print('\nOutput schema:')
    print(FINAL_FIELDS)
    print('\nExample saved row:')
    if flat:
        print(json.dumps(flat[0], ensure_ascii=False, indent=2))


Detected T1 prediction file: /work/Research/MedClaimExtractor/outputs/t1_claim_predictions.jsonl
T1 predicted claim rows: 7564


Generate:   0%|          | 0/946 [00:00<?, ?it/s]

Saved: /work/Research/MedClaimExtractor/outputs/t2/t2_structured_claims.jsonl

Saved final T2 claim-structure output:
/work/Research/MedClaimExtractor/outputs/t2/t2_structured_claims.jsonl

Output schema:
['paper_id', 'sentence_id', 'title', 'year', 'claim_text', 'raw_sentence', 'text', 'model', 'task', 'dataset', 'metric', 'value', 'comparison', 'domain']

Example saved row:
{
  "paper_id": "10.1002/cncr.35732",
  "sentence_id": 5,
  "title": null,
  "year": 2025,
  "claim_text": "All 35 individuals were participants in the University of California, San Francisco Adult Glioma Study.Results: There was a positive correlation among firefighters between the median number of sample variants attributable to single-base substitution signature 42, a single-base substitution mutational signature associated with haloalkane exposure (from the Catalogue of Somatic Mutational Signatures in Cancer) and firefighting years (p = .04; R2 = 0.29).",
  "raw_sentence": "All 35 individuals were participant